<a href="https://colab.research.google.com/github/Akinloyedee/Flexisaf-internship-deliverable1-Advance-machine-techniques-/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Akinloyedee/flyrank_week1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why
**Lane: Ranking Signal Analysis (Lane 1).**

I'm picking this lane because I'm early in this internship and my priority right now is understanding *what actually moves the numbers* before I try to rank, cluster, or predict anything. This lane is EDA-first: correlations, grouped comparisons, and effect sizes rather than a full model. It lets me get comfortable with the starter dataset's gotchas (rate columns, the label trap, missingness patterns) while producing something genuinely useful — a signal report that tells the content team which observed signals are worth paying attention to. It's also the lowest-risk lane to start with: no label-leakage traps like the prediction lanes, and no unsupervised-clustering judgment calls yet. I can always sharpen this into scoring or prediction in later weeks once I trust the signals.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*


**The question:** Which safe, observed content and search signals (position, freshness, engagement, word count, CTR) are actually associated with a content item's visibility, clicks, engagement, or trend direction — and how strong is each association, really?

**Unit of analysis:** one content item (one pseudonymized page), summarized over its trailing 90-day window — the same grain as the starter dataset.

**The decision this improves:** Right now, a content strategist deciding what to review first has to rely on gut feel or a single hand-picked rule (e.g. "just check anything stale"). This work tells them *which* signals reliably line up with declining or thriving content, and how big each effect actually is, so their review criteria are evidence-backed instead of assumed.

**Who acts, and what they do:** A content strategist / SEO lead uses the signal report to decide what to check first when triaging a backlog of pages — e.g., "position tier and freshness show the strongest links to decline, so weight those heavily; word count barely matters here, so stop over-indexing on it."

**Cost of a wrong call:** If the report says a signal matters when it's really just noise, the team spends real review hours chasing a weak lead — and may deprioritize pages that are actually at risk. If it misses a signal that really does matter, they under-prioritize the right pages and a genuinely recoverable page keeps declining unnoticed. Because this project only informs *priority*, not action, the cost of a wrong signal is wasted attention, not a broken production decision.

**Why data/ML helps at all (and not just an if-statement):** A single signal like "CTR by position tier" can be checked with a simple groupby — a plain rule could handle that alone. But real prioritization needs several signals looked at together (position, freshness, engagement, age, word count) and their interactions, and I don't yet know which of those are real versus coincidental in this data. That's exactly the kind of pattern that's too tangled to hand-code confidently — it's worth measuring properly with grouped comparisons and effect sizes rather than assuming.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows) that make Lane 1 worth pursuing:

1. **54.2% of all content rows are currently trending "down"** (`trend_direction == 'down'`, 16,262 of 30,000 rows) — more than half the inventory is moving in the wrong direction. That's a big enough base rate that finding *which* signals travel with decline is genuinely useful, not a rare-event problem.
2. **Mean CTR varies close to 10x across position tiers** — from 0.150% for `deep` positions up to 1.484% for `top_3` — confirming that position is a strong, expected driver of clicks. This is a sanity-check signal (it *should* be strong), and it's a useful baseline against which to compare less obvious signals.
3. **Mean engagement rate differs meaningfully by trend direction**: pages trending "down" average 2.44% engagement versus 2.99% for pages trending "up" — a real but far more modest gap than the CTR-by-position gap. That modesty is exactly why this needs a proper signal audit rather than a guess: the effect is there, but it's not obviously dramatic, and I want to know if it holds up once I control for other factors.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/Akinloyedee/Flyrank-ML-Internship/main/data/raw/content_refresh_anonymized.csv")

print("Total rows:", len(df))

# Number 1: base rate of declining content
trend_counts = df["trend_direction"].value_counts()
trend_pct = (df["trend_direction"].value_counts(normalize=True) * 100).round(1)
print("\ntrend_direction counts:")
print(trend_counts)
print("\ntrend_direction (%):")
print(trend_pct)

# Number 2: CTR by position tier
print("\nMean CTR by position_tier (rate column, already a % per the data dictionary):")
print(df.groupby("position_tier")["ctr"].mean().round(3).sort_values())

# Number 3: engagement rate by trend direction
print("\nMean engagement_rate by trend_direction:")
print(df.groupby("trend_direction")["engagement_rate"].mean().round(3))


Total rows: 30000

trend_direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction (%):
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

Mean CTR by position_tier (rate column, already a % per the data dictionary):
position_tier
deep        0.150
page_3_5    0.222
striking    0.323
page_1      0.652
top_3       1.484
Name: ctr, dtype: float64

Mean engagement_rate by trend_direction:
trend_direction
down      2.437
flat      1.375
new       2.136
stable    2.839
up        2.990
Name: engagement_rate, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can say:**
- Observed associations and effect sizes between specific signals (position, freshness, engagement, etc.) and content performance/trend, measured on this 30,000-row anonymized starter slice (and later the warehouse release).
- Directional patterns: "pages with X tend to also show Y" — described with careful, hedged language ("we observed," "this suggests," "associated with").
- Decision-support guidance: which signals are worth a content strategist's attention when triaging a review backlog.

**What this work can never say:**
- That any signal *causes* a change in rankings, clicks, or engagement — this is observational data, not an experiment, so I cannot claim a refresh (or any content change) caused a recovery.
- That I have proven or measured a Google (or any search engine / AI platform) ranking algorithm factor.
- That a correlation found here generalizes beyond this data slice without re-checking it on the fuller warehouse release with proper validation.
- That a precalculated bucket (like `trend_direction`) is ground truth rather than a derived proxy — I'll keep treating it as a proxy label, not the real world.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.